# Usage Examples

Four end-to-end examples driving the team with `run_research_nohup` — the **detached** entry: runs in its own process (survives a killed kernel), re-run a cell to **reconnect** by the deterministic `run_id`. Plus a monitor-inspection helper. Runtime agents see only `prompts/`; this notebook is for human inspection.

1. **Existing tools** — a question the current `xspec_utils` toolkit already answers (joint broadband spectral fit).
2. **Code authoring (#2)** — a question needing capability the toolkit lacks, so the lead authors the missing tools and workers rerun (Phase 2 step 7).
3. **Code authoring (#7)** — a broader multi-gap survey exercising the same authoring loop on new ground.
4. **Code authoring (#12)** — an independent reproduction of a *published* Cyg X-1 measurement, benchmarking self-derived numbers against Krawczynski+2022 (an anti-fabrication stress: the paper is the target the run must NOT fabricate toward).

In [1]:
from datetime import datetime
import logging
from pathlib import Path
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv(override=True)

for noisy_logger in ("arxiv", "httpcore", "httpx"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

from auto_research import run_research_nohup
from auto_research.schemas import ResearchRequest

## Using existing tools

`chatter=7` sets monitor verbosity; progress streams live into the cell (also persisted to `monitor.jsonl`/`backend.log` under `<runtime_dir>/runs/<run_id>/`). The run is detached — re-run to reconnect; `run.markdown()` shows the grouped report. Answerable as-is via `load_data`, `fit_tbfeo_hss`, `run_fit`, `xspec_status`, `plot_fit`, `plot_iqu`, `group_pha`.

In [ ]:
request = ResearchRequest(
    question=(
            "IXPE has performed a Cyg X-1 observation in its soft state on 2023-05-24. "
            "Find simultaneous or near-simultaneous observations from other X-ray observatories "
            "(e.g. NICER, NuSTAR, XMM-Newton) and perform a joint broadband spectral fit "
            "for this epoch. epoch including observations: IXPE: '02008401', NICER: '6643010101' and '6643010102', NuSTAR: '80902318002'."
        ),
        context=(
            "TEST SESSION — evaluation criteria are relaxed as follows:\n"
            "- Absent NuSTAR and NICER backgrounds are acceptable and expected: "
            "Cyg X-1 is an extremely bright source for which background files are not used "
            "(background contribution is negligible; this is standard practice for bright sources).\n"
            "- The primary goal is to verify that analysts correctly invoked the available "
            "toolkit tools (load_data, fit_tbfeo_hss, run_fit, xspec_status, plot_fit, "
            "plot_iqu, group_pha) and that any inline Python executed via "
            "`mcp.builtin.bash_exec` with `python -c` successfully imports xspec and "
            "xspec_utils without errors.\n"
            "- The existing toolkit will not produce publication-level results. "
            "You may send back one round of revision to analysts, asking for better results. "
            "They should attempt to run inline calculations via python -c until they get "
            "non-error outputs and reasonable-looking fit statistics, "
            "but they are not expected to achieve publication-quality fits.\n"
            "- Accept the analysis if tools were called with correct arguments and returned "
            "non-error outputs, even if fit statistics are not publication-quality.\n"
            "- If any of the tools or environments seemed broken, abort the run immediately and report."
        )
)

# Detached run: survives a killed kernel; re-run this cell to reconnect (run_id = hash of the request).
run = await run_research_nohup(request, chatter=7)

# The detached run's monitor is reconstructed from its on-disk JSONL; save a Markdown copy.
LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/P1/step-12")
LOG_DIR.mkdir(parents=True, exist_ok=True)
md_path = LOG_DIR / f"cyg_x1_ixpe_softstate_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.md"
run.monitor.write_markdown(md_path)
md_path, run.verdict

In [ ]:
display(Markdown(run.markdown()))

## Test code authoring — intra-observation PA/PD stability (#2)

BH-XRB project **#2** (`bhxrb_intraobservation_pa_pd_stability`): time-resolved `xpbin algorithm=PCUBE` (PD/PA/QI/UI/MDP99 in ~5 ks segments). The spectral-only `xspec_utils` manifest exposes no PCUBE / reduction / acquire path, so a worker raises a `tool_request`; the lead authors the missing capability through the gated flow (author → gate → GPT co-lead review → atomic commit on `ai`), then workers rerun. Realistic chain: heasarc acquire → reduction → pcube, bounded by `tool_author_cap`.

**Prereq (out of band):** `research_toolkit` on a clean `ai` branch (preflight asserts it). Escalation via a connector (`'slack'`/`'telegram'`) — detached has no stdin, so `'cli'` is rejected. An honest `failed`/`needs_human` outcome is correct — never fabricated.

In [2]:
authoring_request = ResearchRequest(
    question="""\
IXPE project (P1) — Intra-Observation Polarization Angle and Degree Stability in Bright BH-XRBs.

Targets and datasets:
- Cyg X-1: 01002901, 03002599
- Swift J1727.8-1613: 03005801, 03006001
- 4U 1630-47: 01250401
- LMC X-1: 02001901

This project quantifies the intra-observation stability of both polarization angle (PA) and polarization degree (PD) in long-exposure BH-XRB observations using time-resolved PCUBE analysis. For each observation, we divide the exposure into orbital-timescale segments (~5 ks), measure PA and PD in each segment, and apply chi-squared constancy tests to identify statistically significant variability in either quantity. Because PD is positively biased in the low-significance regime, the model-independent normalized Stokes parameters (Q/I, U/I) provide the statistically robust joint test, while the direct PA and PD constancy tests give the physically interpretable per-quantity assessment. Novel publishable results include: (1) a systematic catalog of PA and PD stability metrics for 6 high-exposure BH-XRB observations spanning 110-563 ks, (2) identification of any observations showing significant PA wander or PD variability beyond statistical noise, and (3) upper limits on PA and PD variability amplitude for sources consistent with constant polarization, constraining the geometric and physical stability of the polarization-producing regions.

Analysis prescription:
STEP 1 - DATA RETRIEVAL: Verify availability of Level-2 event files for all 6 observations: Cyg X-1 (01002901, 03002599), Swift J1727.8-1613 (03005801, 03006001), 4U 1630-47 (01250401), LMC X-1 (02001901). Confirm presence of ixpe_*_det1_evt2.fits, ixpe_*_det2_evt2.fits, ixpe_*_det3_evt2.fits for each obs_id. These observations range from 110 to 563 ks exposure, providing sufficient statistics for time-resolved polarimetry.

STEP 2 - SOURCE REGION DEFINITION AND EVENT EXTRACTION: For each observation:
  (a) Use xselect to define a circular source region of 60 arcsec radius centered on source coordinates.
  (b) Merge DU1, DU2, DU3 event files using xselect into a single combined event list.
  (c) Filter events to 2-8 keV energy range using xselect.
  Output: One filtered, merged event file per observation.

STEP 3 - TIME-RESOLVED POLARIZATION ANALYSIS: For each observation:
  (a) Determine total Good Time Interval (GTI) exposure and divide into N segments of approximately 5 ks each.
  (b) For each time segment, run xpbin with algorithm=PCUBE to generate a polarization cube using NEFF weighting.
  (c) Extract from each PCUBE: PD, PD_err, PA, PA_err, MDP99, and normalized Stokes parameters Q/I, U/I with errors.
  (d) Record segment start time (MJD), stop time, and effective exposure.
  Output per observation: N PCUBE files and a time-series table with columns [segment_id, t_start_mjd, t_stop_mjd, exposure_s, PD_pct, PD_err, PA_deg, PA_err, QI, QI_err, UI, UI_err, MDP99].

STEP 4 - OBSERVATION-INTEGRATED POLARIZATION: For each observation:
  (a) Run xpbin with algorithm=PCUBE on the full merged event file to obtain observation-averaged PD, PA, Q/I, U/I with errors.
  (b) Compute MDP99 for the full observation.
  (c) Record detection significance as PD/sigma_PD.
  Output: Integrated polarization parameters for each observation.

STEP 5 - PA AND PD CONSTANCY TESTS: For each observation:
  (a) Compile the time-series of PA and PD measurements from Step 3.
  (b) PA constancy: perform weighted chi-squared test chi2_PA = sum[(PA_i - PA_mean)^2 / PA_err_i^2], where PA_mean is the weighted mean PA.
  (c) PD constancy: perform weighted chi-squared test chi2_PD = sum[(PD_i - PD_mean)^2 / PD_err_i^2], where PD_mean is the weighted mean PD.
  (d) Compute degrees of freedom dof = N_segments - 1 for each test.
  (e) Calculate p-values from chi2 and dof using scipy.stats.chi2.sf for both PA and PD.
  (f) Flag observation as showing significant PA variability if p_value_PA < 0.01, and significant PD variability if p_value_PD < 0.01 (99% confidence).
  (g) CAVEAT: PD is a positively-biased quantity in the low-significance regime (PD/PD_err < ~3) and its distribution departs from Gaussian, so the chi2_PD test is only approximate when segment significances are low. Interpret chi2_PD alongside the model-independent Q/I, U/I constancy test of Step 6, which is unbiased and is the statistically robust joint reference.
  Output per observation: chi2_PA, chi2_PD, dof, p_value_PA, p_value_PD, pa_variability_flag, pd_variability_flag.

STEP 6 - STOKES PARAMETER CONSTANCY TEST (UNBIASED JOINT REFERENCE): For each observation:
  (a) Perform weighted chi-squared test for constancy of Q/I time series: chi2_QI = sum[(QI_i - QI_mean)^2 / QI_err_i^2].
  (b) Perform weighted chi-squared test for constancy of U/I time series: chi2_UI = sum[(UI_i - UI_mean)^2 / UI_err_i^2].
  (c) Compute combined 2D chi-squared: chi2_2D = chi2_QI + chi2_UI with dof_2D = 2*(N_segments - 1).
  (d) Calculate p-values for each test. Because Q/I and U/I are approximately Gaussian and unbiased, the 2D test is the primary, model-independent verdict on whether the polarization vector (PA and PD jointly) is constant.
  Output per observation: chi2_QI, chi2_UI, chi2_2D, p_value_QI, p_value_UI, p_value_2D.

STEP 7 - PA AND PD VARIABILITY AMPLITUDE QUANTIFICATION: For each observation:
  (a) Compute standard deviation of measured PA values (sigma_PA_measured) and of measured PD values (sigma_PD_measured).
  (b) Compute expected statistical scatter: sigma_PA_expected = sqrt(mean(PA_err_i^2)) and sigma_PD_expected = sqrt(mean(PD_err_i^2)).
  (c) If sigma_measured > sigma_expected for a quantity, compute intrinsic scatter sigma_intrinsic = sqrt(sigma_measured^2 - sigma_expected^2), separately for PA and PD.
  (d) Otherwise, report sigma_intrinsic as an upper limit (sigma_intrinsic < sigma_expected), separately for PA and PD.
  (e) Compute peak-to-peak ranges: PA_range = PA_max - PA_min and PD_range = PD_max - PD_min.
  Output: sigma_PA_intrinsic (or upper limit), sigma_PD_intrinsic (or upper limit), PA_range, PD_range.

STEP 8 - STRUCTURE FUNCTION ANALYSIS: For each observation with N >= 10 segments:
  (a) Compute first-order structure function of PA: SF_PA(tau) = <(PA(t+tau) - PA(t))^2>, and of PD: SF_PD(tau) = <(PD(t+tau) - PD(t))^2>.
  (b) Compute each SF at lag tau = 5 ks, 10 ks, 20 ks (if exposure permits).
  (c) Test for any trend in SF with lag that would indicate correlated PA or PD wandering.
  Output: PA and PD structure function values at each lag.

STEP 9 - CROSS-SOURCE COMPARISON: Compile results from all 6 observations into summary table.
  (a) Rank observations by PA constancy p-value and, separately, by PD constancy p-value to identify the most/least stable sources in each quantity.
  (b) Compute correlation between integrated MDP99 and chi2_PA/dof, and between MDP99 and chi2_PD/dof, using Spearman rank to check whether apparent variability is driven by measurement precision.
  (c) Compare mean intrinsic PA scatter and mean intrinsic PD scatter (or upper limits) between sources.
  (d) Identify whether any source shows variability in one quantity but not the other (e.g., stable PA with variable PD, or vice versa), which would help distinguish purely geometric reorientation from changes in scattering optical depth / emission-region physics.

STEP 10 - VISUALIZATION: Generate diagnostic and publication-quality figures using matplotlib:
  (a) Time-series plots for each observation with stacked panels sharing a common time axis: PA vs time and PD vs time, with error bars, overplotting the weighted mean as a horizontal line in each panel.
  (b) Q/I vs U/I Stokes plane plots for each observation showing all time segments as points with error ellipses, color-coded by time (this jointly visualizes PA and PD evolution).
  (c) Histograms of reduced chi-squared values (chi2_PA/dof and chi2_PD/dof) across all observations.
  (d) Summary figures: PA scatter vs observation duration and PD scatter vs observation duration for all sources.
  (e) Structure function plots SF_PA(tau) and SF_PD(tau) for observations with sufficient segments.

STEP 11 - OUTPUT PRODUCTS:
  (a) Master results table: pa_pd_stability_catalog.csv with columns [source, obs_id, exposure_ks, n_segments, PD_integrated, PD_integrated_err, PA_integrated, PA_integrated_err, chi2_PA, p_value_PA, chi2_PD, p_value_PD, dof, chi2_2D, p_value_2D, sigma_PA_intrinsic, sigma_PD_intrinsic, PA_range_deg, PD_range_pct, pa_variability_flag, pd_variability_flag].
  (b) Time-series data files per observation: {obs_id}_polarization_timeseries.csv with per-segment PA and PD measurements.
  (c) Summary statistics: pa_pd_variability_summary.txt reporting which observations pass/fail the PA and PD constancy tests at 99% confidence.
  (d) Publication-quality figures: pa_pd_timeseries_all.pdf, stokes_plane_evolution.pdf, chi2_histograms.pdf, structure_function.pdf.
""",
    context="""\
TEST SESSION — this exercises lead-driven tool authoring; evaluate the loop, not publication quality.
- All listed observations are IXPE observations (mission = "ixpe").
- Data is local-first.
- When a worker finds a capability missing, it must raise a tool_request (never fabricate numbers); the lead browses research_toolkit/lib read-only, decides extend-vs-new, and authors the missing tool(s) in dependency order (acquire -> reduce -> pcube). Each authored tool is gated and GPT co-lead reviewed before an atomic commit on branch 'ai'; the workers then rerun on it.
- Evaluation: verify the authoring loop runs end-to-end and the analysis completes on a freshly authored tool. An honest failed/needs_human outcome (e.g. a tool that cannot be made mergeable within the rework caps) is an acceptable, correct result — do NOT fabricate a polarization measurement. If the environment is broken, abort and report.
""",
)

# Detached authoring run: survives a killed kernel; re-run to reconnect. A detached child has no
# stdin, so human_channel='cli' is rejected — the lead/author escalate via a connector ('slack'/'telegram').
authoring_run = await run_research_nohup(authoring_request, chatter=7, human_channel="slack")

AUTHORING_LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/P2/step-7")
AUTHORING_LOG_DIR.mkdir(parents=True, exist_ok=True)
authoring_md = AUTHORING_LOG_DIR / f"p1_pa_pd_stability_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.md"
authoring_run.monitor.write_markdown(authoring_md)
authoring_md, authoring_run.verdict


  [preflight] required_env … ok
  [preflight] database_health … ok
  [preflight] author_reviewer_decorrelation … ok
  [preflight] toolkit_root_configured … ok
  [preflight] toolkit_git_identity … ok
  [preflight] toolkit_branch_clean … ok
  [preflight] catalog_health …

 ok
  [preflight] human_channel …⏳ Launch preflight: a confirmation ping was sent to your human channel — reply to it to start the run (waiting up to 60s).
 ok
[06:54:26] #0001 run L1 run.start: Research run started | run_id=46ed66714843
[06:54:27] #0002 librarian L3 librarian.literature.start: Literature node started
[06:54:33] #0003 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_call librarian_io: librarian_io calling search_literature | tool=search_literature status=ok
[06:54:35] #0004 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_result librarian_io: librarian_io tool search_literature returned | tool=search_literature status=ok chars=4251 figures=0
[06:54:35] #0005 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_call librarian_io: librarian_io calling search_literature | tool=search_literature status=ok
[06:54:39] #0006 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_result librarian_io: librarian_i

[07/02/26 02:32:43] INFO     Retrying request to /v1/messages in 0.396609 seconds              ]8;id=12892857;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/anthropic/_base_client.py\_base_client.py]8;;\:]8;id=12892858;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/anthropic/_base_client.py#1816\1816]8;;\

[07:32:47] #0370 analyst(claude-sonnet-4-6) L7 analyst_worker.analyze.tool_call analyst_worker: analyst_worker calling mcp_builtin_bash_exec | tool=mcp_builtin_bash_exec status=ok
[07:32:47] #0371 analyst(claude-sonnet-4-6) L7 analyst_worker.analyze.tool_result analyst_worker: analyst_worker tool mcp_builtin_bash_exec returned | tool=mcp_builtin_bash_exec status=ok chars=9 figures=0
[07:32:47] #0372 analyst(claude-sonnet-4-6) L7 analyst_worker.analyze.tools analyst_worker: analyst_worker ran 1 tool(s) (iteration 80) | tools=mcp_builtin_bash_exec
[07:32:50] #0373 analyst(claude-sonnet-4-6) L7 analyst_worker.analyze.tool_call analyst_worker: analyst_worker calling mcp_builtin_bash_exec | tool=mcp_builtin_bash_exec status=ok
[07:32:50] #0374 analyst(claude-sonnet-4-6) L7 analyst_worker.analyze.tool_result analyst_worker: analyst_worker tool mcp_builtin_bash_exec returned | tool=mcp_builtin_bash_exec status=ok chars=9 figures=0
[07:32:50] #0375 analyst(claude-sonnet-4-6) L7 analyst_worker.

[07/02/26 02:46:11] INFO     Using bundled Claude Code CLI:                                   ]8;id=12892865;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/claude_agent_sdk/_internal/transport/subprocess_cli.py\subprocess_cli.py]8;;\:]8;id=12892866;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/claude_agent_sdk/_internal/transport/subprocess_cli.py#124\124]8;;\
                             /grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packag                      
                             es/claude_agent_sdk/_bundled/claude                                                   

...

[07/02/26 03:19:28] INFO     Retrying request to /v1/messages in 0.412344 seconds              ]8;id=12892871;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/anthropic/_base_client.py\_base_client.py]8;;\:]8;id=12892872;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/anthropic/_base_client.py#1816\1816]8;;\

[08:21:37] #0911 analyst L7 analyst.worker.initial_report worker-1: Analyst worker 1 produced initial report
[08:21:42] #0912 analyst(gpt-5.5) L7 analyst_worker.analyze.tool_call analyst_worker: analyst_worker calling data_catalog | tool=data_catalog status=ok
[08:21:42] #0913 analyst(gpt-5.5) L7 analyst_worker.analyze.tool_result analyst_worker: analyst_worker tool data_catalog returned | tool=data_catalog status=ok chars=1054 figures=0
[08:21:42] #0914 analyst(gpt-5.5) L7 analyst_worker.analyze.tools analyst_worker: analyst_worker ran 1 tool(s) (iteration 1) | tools=data_catalog
[08:21:47] #0915 analyst(gpt-5.5) L7 analyst_worker.analyze.tool_call analyst_worker: analyst_worker calling mcp_builtin_bash_exec | tool=mcp_builtin_bash_exec status=ok
[08:21:47] #0916 analyst(gpt-5.5) L7 analyst_worker.analyze.tool_result analyst_worker: analyst_worker tool mcp_builtin_bash_exec failed | tool=mcp_builtin_bash_exec status=error chars=205 figures=0
[08:21:47] #0917 analyst(gpt-5.5) L7 analys

[07/02/26 03:32:09] INFO     Using bundled Claude Code CLI:                                   ]8;id=12892877;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/claude_agent_sdk/_internal/transport/subprocess_cli.py\subprocess_cli.py]8;;\:]8;id=12892878;file:///grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packages/claude_agent_sdk/_internal/transport/subprocess_cli.py#124\124]8;;\
                             /grad/fzhou/.conda/envs/auto_research/lib/python3.13/site-packag                      
                             es/claude_agent_sdk/_bundled/claude                                                   

[08:32:10] #0990 tooling L4 coder.session_start coder: authoring ixpe_pa_pd_stability_plot on branch ai | tool=ixpe_pa_pd_stability_plot status=ok
[08:32:12] #0991 tooling L7 coder.tool_use coder: coder → Read | tool=Read status=ok
[08:32:13] #0992 tooling L7 coder.tool_result coder: coder tool result
[08:32:13] #0993 tooling L7 coder.tool_use coder: coder → Read | tool=Read status=ok
[08:32:13] #0994 tooling L7 coder.tool_result coder: coder tool result
[08:32:13] #0995 tooling L7 coder.tool_use coder: coder → Read | tool=Read status=ok
[08:32:13] #0996 tooling L7 coder.tool_result coder: coder tool result
[08:32:17] #0997 tooling L7 coder.tool_use coder: coder → Read | tool=Read status=ok
[08:32:17] #0998 tooling L7 coder.tool_result coder: coder tool result
[08:32:18] #0999 tooling L7 coder.tool_use coder: coder → Read | tool=Read status=ok
[08:32:18] #1000 tooling L7 coder.tool_result coder: coder tool result
[08:32:18] #1001 tooling L7 coder.tool_use coder: coder → Bash | tool=Bas

(PosixPath('/data/fzhou/auto_research/logs/dev/P2/step-7/p1_pa_pd_stability_20260702T015350_monitor.jsonl'),
 PosixPath('/data/fzhou/auto_research/logs/dev/P2/step-7/p1_pa_pd_stability_20260702T015350_monitor.md'),
 ReviewVerdict(accepted=True, rationale='This is an explicit TEST SESSION whose evaluation criterion is that the lead-driven tool-authoring loop runs end-to-end and the analysis completes on freshly authored tools without fabrication; I verified all of that from the tool results. (1) Authoring loop end-to-end: I read /nfsdata1/fzhou/research_toolkit/xspec_utils/lib/pa_pd_stability.py (committed in the toolkit) and confirmed it is a pure numpy/scipy/matplotlib module importing neither xspec nor ixpeobssim, implementing exactly the prescribed statistics — inverse-variance weighted chi2 for PD/QI/UI (_weighted_stat), mod-180 circular PA chi2 via doubled-angle mean with _wrap180 residual wrapping (_weighted_stat_pa), the authoritative 2D Stokes test chi2_2D=chi2_QI+chi2_UI with 

In [ ]:
display(Markdown(authoring_run.markdown()))

## Test code authoring — broadband PD–hardness survey (#7)

BH-XRB project **#7** (`bhxrb_broadband_pd_hardness_survey`): 29 IXPE obs of 6 BH-XRBs correlating broadband (2–8 keV) PD against spectral hardness across states. Exercises the authoring loop on **new ground** vs P1 — needs broadband PCUBE, **hardness ratios from background-subtracted Stokes-I / PHA spectra** (a new reduction path), and survey stats (Spearman, KS two-sample, Mann-Whitney U, optional censored Kendall tau). Workers hit several gaps → the lead authors tools in dependency order (acquire → reduce → pcube/spectra → stats → plot) through the gated flow, then rerun.

Spanning **many** gaps, this is the motivating case for TODO **[A8]** (batch gaps → lead drives the whole chain before re-running analysts) — look into [A8] before launching.

**Prereq (out of band):** clean `ai` branch (preflight asserts it). Escalation via **Telegram** (`'cli'` rejected on a detached child). Honest `failed`/`needs_human` is correct — never fabricated.

In [3]:
survey_request = ResearchRequest(
    question="""\
BH-XRB project (#7) — Broadband Polarization-Hardness Correlation Survey Across BH-XRBs.

Targets and datasets (all IXPE, mission = "ixpe"; 29 observations across 6 sources):
- Cyg X-1: 01002901, 01250101, 02008201, 02008301, 02008401, 02008501, 02008601, 03002201, 03003101, 03010001, 03010101, 03002599
- Swift J1727.8-1613: 02250901, 02251001, 02251101, 02251201, 02251301, 03005701, 03006001, 03005801
- LMC X-1: 02001901
- LMC X-3: 02006599, 03004899, 03004901, 03005001
- 4U 1630-47: 01250401, 02250601
- GX 339-4: 03005101, 03005301

This project surveys the correlation between broadband (2-8 keV) X-ray polarization degree and spectral hardness ratio across 29 observations of 6 BH-XRBs spanning diverse spectral states. Using model-independent PCUBE analysis for PD/PA and background-subtracted PHA spectra for hardness ratios, we test whether PD systematically correlates with HR within sources (intra-source variability) and across the sample (cross-source trends). Sources are selected to span persistent hard-state systems (Cyg X-1), transient outbursts (Swift J1727.8-1613, 4U 1630-47, GX 339-4), and soft-state-dominated systems (LMC X-1, LMC X-3), enabling comparison of polarization behavior across spectral states. Spearman rank correlations quantify PD-HR relationships, while KS tests compare PD distributions between hard and soft spectral states. This systematic cross-source survey of PD-hardness correlations using validated background subtraction has not been published and would establish whether spectral state is a primary driver of X-ray polarization in BH-XRBs.

Analysis prescription:
STEP 1 - DATA RETRIEVAL: Verify all 29 obs_ids exist in the local Level-2 archive. Locate all DU event files (evt2 files) for each observation. Decompress .fits.gz files to a working directory if needed. Abort if any dataset is missing.

STEP 2 - SOURCE AND BACKGROUND REGION DEFINITION: For each source, define a standard circular source extraction region (r=60 arcsec centered on source coordinates) and an annular background region (inner radius 180 arcsec, outer radius 240 arcsec) following BH-XRB consensus from the literature. Store region definitions in DS9 format for reproducibility.

STEP 3 - BROADBAND PCUBE EXTRACTION: For each observation, run ixpeobssim pipeline.xpbin on all three DU event files with algorithm='PCUBE', emin=2.0, emax=8.0 keV, explicitly set weights='NEFF'. Do NOT specify irfname; let ixpeobssim auto-resolve from event headers. Read the output PCUBE FITS file using ixpeobssim.binning.polarization.xBinnedPolarizationCube. Extract: PD, PD_err (1-sigma), PA, PA_err, MDP99. Flag observations with MDP99 > 15% as having insufficient sensitivity.

STEP 4 - STOKES I SPECTRA EXTRACTION FOR HARDNESS RATIOS: For each observation and each DU, extract Stokes I spectra from source and background regions (xpbin algorithm='PHA1', or xselect). Compute soft-band (2-4 keV) and hard-band (4-8 keV) source rates minus area-scaled background (scale_factor = source_area / background_area). Sum across all three DUs for combined rates.

STEP 5 - HARDNESS RATIO CALCULATION: Compute HR = (Hard_rate - Soft_rate) / (Hard_rate + Soft_rate) from background-subtracted count rates. Propagate errors using Poisson statistics and ratio error propagation. Record HR and HR_err for each observation.

STEP 6 - DETECTION CLASSIFICATION: Classify each observation: 'Detected' if PD > MDP99 (>99% confidence), else 'Non-detection'. For non-detections record the upper limit as MDP99 (standard IXPE convention); do not use the biased PD measurement for non-detections in correlation analyses.

STEP 7 - INTRA-SOURCE SPEARMAN CORRELATION (DETECTIONS ONLY): For sources with >=4 detections, compute Spearman rho and p-value for PD vs HR (scipy.stats.spearmanr); flag significant if p < 0.05. Record N_detections, rho, p-value.

STEP 8 - INTRA-SOURCE ANALYSIS INCLUDING UPPER LIMITS (SURVIVAL ANALYSIS): For sources with mixed detections/non-detections, use Kendall tau with censoring (lifelines/survival tools if available) treating upper limits as censored. If unavailable, run (1) detections-only Spearman and (2) Mann-Whitney U on HR between detected vs non-detected.

STEP 9 - CROSS-SOURCE COMBINED ANALYSIS: Combine all secure detections across the six sources. (a) Spearman PD vs HR for the combined dataset. (b) Group by dominant state: soft (LMC X-1, LMC X-3) vs hard/intermediate (Cyg X-1, Swift J1727, 4U 1630-47, GX 339-4). (c) Two-sample KS test (scipy.stats.ks_2samp) comparing PD distributions between groups. (d) Mann-Whitney U comparing HR between detected/non-detected to assess selection effects.

STEP 10 - FEASIBILITY ASSESSMENT: Tabulate per observation: exposure (ks), MDP99, detection status. Summarize N_obs per source, N_detections per source, exposure range per source.

STEP 11 - VISUALIZATION: (a) PD vs HR scatter for all detections, color-coded by source, error bars, non-detections as downward arrows at MDP99. (b) Per-source panel figure of PD vs HR with detections and upper limits. (c) Box plots of PD distributions, soft vs hard groups (detections only). (d) HR histogram for detected vs non-detected. (e) Feasibility plot MDP99 vs exposure with detection-threshold line.

STEP 12 - OUTPUT PRODUCTS: bhxrb_pd_hardness_survey.csv (source, obs_id, exposure_ks, PD, PD_err, PA, PA_err, MDP99, detected, HR, HR_err); intra_source_correlations.csv; combined_correlation.csv; ks_test_soft_vs_hard.csv; detection_bias_assessment.csv; figures pd_vs_hr_combined.pdf, pd_vs_hr_per_source.pdf, pd_distribution_comparison.pdf, hr_detection_bias.pdf, feasibility_mdp99_exposure.pdf.

Relevant literature: Krawczynski et al. (arXiv:2206.09972); Veledina et al., Swift J1727.8-1613 (arXiv:2310.07681); Krawczynski et al., Cas A (arXiv:2206.06713).
""",
    context="""\
TEST SESSION — this exercises lead-driven tool authoring; evaluate the loop, not publication quality.
- All listed observations are IXPE observations (mission = "ixpe"). Data is local-first.
- New capabilities this survey needs beyond the spectral-only toolkit: broadband (2-8 keV) PCUBE PD/PA/MDP99; hardness ratios from background-subtracted Stokes-I / PHA spectra (source + annular background, area scaling); and correlation statistics (Spearman, KS two-sample, Mann-Whitney U; optional censored Kendall tau). When a worker finds a capability missing, it must raise a tool_request (never fabricate numbers); the lead browses research_toolkit/lib read-only, decides extend-vs-new, and authors the missing tool(s) in dependency order (acquire -> reduce -> pcube/spectra -> stats -> plot). Each authored tool is gated and GPT co-lead reviewed before an atomic commit on branch 'ai'; the workers then rerun on it.
- Evaluation: verify the authoring loop runs end-to-end and the analysis completes on freshly authored tools. An honest failed/needs_human outcome (a tool that cannot be made mergeable within the rework caps) is an acceptable, correct result — do NOT fabricate a polarization measurement or a correlation. If the environment is broken, abort and report.
""",
)

# Detached authoring run: survives a killed kernel; re-run to reconnect. A detached child has no
# stdin, so human_channel='cli' is rejected — the lead/author escalate via Telegram.
survey_run = await run_research_nohup(survey_request, chatter=7, human_channel="telegram")

SURVEY_LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/P7-broadband/step-7")
SURVEY_LOG_DIR.mkdir(parents=True, exist_ok=True)
survey_md = SURVEY_LOG_DIR / f"p7_broadband_pd_hardness_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.md"
survey_run.monitor.write_markdown(survey_md)
survey_md, survey_run.verdict


[15:05:21] #0001 run L1 run.start: Research run started | run_id=a36ac72ce11c
[15:05:21] #0002 librarian L3 librarian.literature.start: Literature node started
[15:05:28] #0003 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_call librarian_io: librarian_io calling search_literature | tool=search_literature status=ok
[15:05:30] #0004 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_result librarian_io: librarian_io tool search_literature returned | tool=search_literature status=ok chars=4192 figures=0
[15:05:30] #0005 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_call librarian_io: librarian_io calling search_literature | tool=search_literature status=ok
[15:05:34] #0006 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_result librarian_io: librarian_io tool search_literature returned | tool=search_literature status=ok chars=12 figures=0
[15:05:34] #0007 librarian(claude-sonnet-4-6) L7 librarian_io.gather_lit

(PosixPath('/data/fzhou/auto_research/logs/dev/P7-broadband/step-7/p7_broadband_pd_hardness_20260707T102430_monitor.md'),
 ReviewVerdict(accepted=True, rationale="This is an explicitly-flagged toolkit-loop test, and the analyst report is an honest `ok` outcome that completed end-to-end on real tool output with no fabricated numbers. I verified the survey module directly: research_toolkit/xspec_utils/lib/bhxrb_pd_hr_survey.py is present and clean \reindependent _read_pcube_integrated of the segment_id=-1 provenance convention, detections-only Spearman, a caller-supplied (never hardcoded) soft/hard group split for the KS test, and NaN-guards on all four statistics so an empty/short group returns NaN rather than raising. Both workers independently confirmed the full chain (ixpe_reduce_pha, ixpe_HR->bhxrb_hardness_ratios.csv, bhxrb_pd_hr_survey, pd_hr_survey_plot, pcube_timeseg integrated PD) already exists, so no tool_request and no authoring was warranted \re the requested censored-Kenda

In [ ]:
display(Markdown(survey_run.markdown()))

## Reproduce Krawczynski+2022 — Cyg X-1 polarization (#12)

BH-XRB project **#12** (`cygx1_reproduce_krawczynski2022_polarization`): independently re-derive the IXPE 2–8 keV polarization of Cyg X-1 (May 2022 `01002901` + June 2022 ToO `01250101`) and benchmark against Krawczynski et al. 2022 (Science) Tables S1–S2. Run as a **real production** request — the team decides for itself whether the toolkit suffices or new capability must be authored; nothing here pre-identifies the gaps.

Reproduction integrity: a *published* answer exists, so the paper is a **benchmark to compare against, never a target to match** — report the real measured values and flag any >3σ discrepancy honestly.

**Prereq (out of band):** clean `ai` branch (preflight asserts it). Escalation via **Telegram** (`'cli'` rejected on a detached child). Honest `failed`/`needs_human` is a correct outcome — never fabricated.

In [2]:
repro_request = ResearchRequest(
    question="""\
Cyg X-1 project (#12) — Independent Reproduction of Cyg X-1 May-2022 IXPE Polarization Measurements.

Targets and datasets (IXPE, mission = "ixpe"):
- Cyg X-1: 01002901 (2022 May, ~241 ks), 01250101 (2022 June ToO, ~86 ks)

We independently re-derive the IXPE polarization of Cygnus X-1 reported by Krawczynski et al. (2022, Science) using the same Level-2 data (May 2022 obs 01002901 and June 2022 ToO 01250101) and benchmark our numbers against their Tables S1-S2. With the standard ixpeobssim PCUBE analysis and an XSPEC Stokes I,Q,U cross-check (tbabs*polconst*powerlaw) we obtain the 2-8 keV polarization degree (PD), angle (PA), 1-sigma errors, MDP99, significance, and the energy-resolved PD(E)/PA(E) spectra. The deliverable is a quantitative this-work-vs-paper agreement table with each difference expressed in sigma, confirming the robustness of the headline hard-state coronal polarization and the reported PD-increase-with-energy. The forbidden corona-geometry/ray-tracing modelling of the original paper is explicitly NOT reproduced; only standard-measurement polarization products are.

Analysis prescription:
STEP 1 - DATA RETRIEVAL & VERIFICATION. Locate Level-2 event files for Cyg X-1 obs_id 01002901 (2022 May, ~241 ks) and 01250101 (2022 June ToO, ~86 ks) in the local IXPE archive (three DU files per obs, *.fits.gz). Verify all six files exist; ABORT with a clear FAILED report if any is missing (do NOT fabricate). Read each event header to confirm OBJECT matches Cyg X-1, extract RA/DEC, TSTART/TSTOP, MJDREFI/MJDREFF, and response/IRF strings. Decompress to a working directory if any tool rejects gzipped input.

STEP 2 - SMOKE TEST (one DU, one obs). Import ixpeobssim.core.pipeline. On a single DU file of 01002901 run xpbin algorithm='PCUBE', emin=2.0, emax=8.0, ebins=1, letting irfname auto-resolve from headers (irfname=None); if auto-resolution fails, glob installed MRFs under <ixpeobssim>/caldb/ixpe/gpd/cpf/mrf and pick the obssim<DATE>:vNN (zero-padded, e.g. v013) matching the 2022 epoch and weighting flavor. Read the output with ixpeobssim.binning.polarization.xBinnedPolarizationCube and confirm non-empty PD/PA/MDP99. Use help() to confirm at runtime the exact accessor names for PD, PA (and errors), and MDP99. Record them; all later steps use these accessors, NOT hand-rolled numpy arithmetic on raw Q/U.

STEP 3 - REGION & WEIGHTING SETUP (match the paper, CORRECT UNITS). Paper used a circular source region radius ~80 arcsec and a concentric background annulus inner ~150 / outer ~310 arcsec. CRITICAL: ixpeobssim xpselect --rad/--innerrad are in ARCMINUTES. Convert: source --rad=80/60=1.33 arcmin; annulus --innerrad=150/60=2.50, --rad=310/60=5.17 arcmin. Use NEFF-weighted events (xpbin acceptcorr=True, weights=True). For EACH DU of EACH obs run xpselect TWICE on the same event file: (a) SOURCE --ra <src_ra> --dec <src_dec> --rad 1.33; (b) BACKGROUND --ra --dec --rad 5.17 --innerrad 2.50. Keep source and background event files separate per DU. Compute the extraction AREAS: A_src = pi*1.33^2, A_bkg = pi*(5.17^2 - 2.50^2) arcmin^2 (area ratio ~11.5). To verify the paper's >70x source/background claim, compare AREA-NORMALIZED surface brightness: divide each region's 2-8 keV count rate by its area, then take the ratio (source-rate/A_src) / (bkg-rate/A_bkg). Report this corrected ratio (expected ~70x); do NOT compare raw counts or unscaled rates.

STEP 4 - BACKGROUND-SUBTRACTED PCUBE (native ixpeobssim background tooling). For 01002901, in a single 2-8 keV bin, produce a SOURCE polarization cube and a BACKGROUND polarization cube with xpbin algorithm='PCUBE' (emin=2.0, emax=8.0, ebins=1, acceptcorr=True, weights=True): pass the three per-DU SOURCE event files to one xpbin call (DU combination handled natively) and the three per-DU BACKGROUND event files to a second xpbin call. Subtract the background at the Stokes level using ixpeobssim's NATIVE background machinery: run the ixpeobssim background-subtraction utility (xpbkgsubtract / the xBinnedPolarizationCube subtraction supported by the framework) supplying the SOURCE and BACKGROUND cubes and the geometric BACKSCAL = A_src/A_bkg so the background is correctly area-scaled before subtraction with modulation-factor-aware error propagation. Do NOT search the cube-reader object for a method that does not exist; confirm the correct entry point in the smoke test via --help/help(). Read PD, PA (1-sigma errors), MDP99 and significance from the background-subtracted cube via the accessors. If the native subtraction utility cannot be run, report this step as FAILED and rely on the XSPEC BACKFILE route (STEP 7) for the background-subtracted measurement.

STEP 5 - BROADBAND 2-8 keV POLARIZATION (primary deliverable). From STEP 4 report the background-subtracted DU-combined 2-8 keV PD, PA (1-sigma), q=Q/I, u=U/I, MDP99 and significance for 01002901 (paper: PD=4.01+/-0.20%, PA=-20.7+/-1.4 deg, ~20 sigma). Also produce per-DU (det1/det2/det3) background-subtracted cubes by running the STEP-4 procedure on each DU's source/background pair separately, and read PD, PA, q, u, significance per DU to compare with paper Table S1 (Q/I, U/I) and Table S2 (PD, PA, significance).

STEP 6 - ENERGY-RESOLVED POLARIZATION (PD(E), PA(E)). Re-run the STEP-4 source/background PCUBE procedure with the paper's four energy bands using ebinalg=LIST and ebinning edges [2,3,4,6,8] keV, and separately with the EXACT grid that reproduces the paper's fit statistics: the paper reports chi2=4.04 for 9 DOF on a 2-parameter linear fit, i.e. 9+2 = 11 energy bins. Use 11 equal-width bins of 6/11 keV across 2-8 keV (edges = 2 + k*6/11 for k=0..11). For each band read background-subtracted PD, PA, errors, significance from the cube accessors (DU-combined). Tabulate the four-band results against Table S2. Fit the 11-bin PD(E) with scipy.optimize.curve_fit using (a) a constant and (b) a linear model PD = A + B*(E/keV - 1); report A, B, errors, chi2/dof (linear dof = 11-2 = 9) for both, and the constant-model rejection probability (chi2 survival function). Compare to the paper's A=(2.9+/-0.4)%, B=(0.58+/-0.15)%, linear chi2=4.04/9 (p=0.909), and the 99.93% constant-model rejection (~3.4 sigma PD-increase-with-energy). Also fit PA(E) with a constant and report its chi2/dof and p-value (paper p=0.575).

STEP 7 - XSPEC SPECTROPOLARIMETRIC CROSS-CHECK (native BACKFILE background subtraction). For 01002901 build IXPE Stokes I, Q, U spectra for EACH DU using xpbin algorithms PHA1, PHA1Q, PHA1U on BOTH the SOURCE and BACKGROUND event files from STEP 3. Generate/associate CALDB ARF/MRF/RMF (ixpecalcarf for the source region; NEFF weighting). CRITICAL BACKSCAL: ixpeobssim PHA1/PHA1Q/PHA1U does NOT write a correct BACKSCAL for an annulus, so before loading into XSPEC use a HEASoft header editor (fthedit / fparkey) to write BACKSCAL = A_src into each SOURCE PHA header and BACKSCAL = A_bkg into each BACKGROUND PHA header (consistent units), so XSPEC area-scales the background by A_src/A_bkg rather than 1:1. In PyXSPEC load the three DUs' I,Q,U source spectra, attach the matching background spectra via BACKFILE for each of I, Q, U, and fit tbabs*polconst*powerlaw over 2-8 keV (fix NH~0.44e22 if needed for stability; tie polconst PD and PA across DUs). Read the 2-8 keV PD and PA with 68.3% errors from the polconst parameters and compare to PCUBE and to the paper's XSPEC column (PD=3.9+/-0.2%, PA=-21+/-1 deg).

STEP 8 - JUNE ToO & COMBINED CHECK (native combination). Repeat STEP 3-4 (broadband 2-8 keV background-subtracted PCUBE) for 01250101 (June); report DU-combined PD, PA, errors. To combine May+June, do NOT sum Stokes counts by hand: pass ALL six source event files (3 DU x 2 obs) to one xpbin PCUBE call and ALL six background event files to a second, then apply the STEP-4 native background subtraction (with the same BACKSCAL=A_src/A_bkg) to obtain the combined PD, PA, errors from the cube accessors. As independent confirmation, also load the May and June I,Q,U source+BACKFILE spectra together in one PyXSPEC session and fit a single tied polconst. Compare to the paper's June PD=3.84+/-0.31%, PA=-25.7+/-2.3 deg and combined PD=3.95+/-0.17%, PA=-22.2+/-1.2 deg.

STEP 9 - COMPARISON TABLE (final deliverable). Build a CSV of this-work vs paper-reference vs difference and significance = |this-paper| / sqrt(sigma_this^2 + sigma_paper^2) for every quantity: 2-8 keV PD, PA (May, June, combined); the four energy-band PD/PA; the per-DU 2-8 keV Q/I,U/I,PD,PA (Tables S1/S2); the linear-fit A,B; and the XSPEC cross-check PD/PA. Flag any quantity discrepant at >3 sigma and discuss likely causes (CALDB version, weighting flavor, region size). Save all PCUBE FITS, Stokes spectra and responses, fit results, the comparison CSV, a Figure-S5-style PD(E)/PA(E) plot, and a Figure-S3-style q-u plane plot to the project output directory. If any real tool step cannot run, report it as FAILED rather than substituting synthetic values.

Relevant literature: Krawczynski, H. et al. — Polarized X-rays constrain the disk-jet geometry in the black hole X-ray binary Cygnus X-1 (https://ui.adsabs.harvard.edu/abs/2022Sci...378..650K); Baldini, L. et al. — ixpeobssim: A simulation and analysis framework for the Imaging X-ray Polarimetry Explorer (https://ui.adsabs.harvard.edu/abs/2022SoftX..1901194B); Weisskopf, M. C. et al. — Imaging X-ray Polarimetry Explorer: prelaunch (https://ui.adsabs.harvard.edu/abs/2022JATIS...8b6002W)
""",
    context="""\
- Both observations are IXPE observations (mission = "ixpe"). Data is local-first: Cyg X-1 01002901 (2022 May) and 01250101 (2022 June ToO).
- This is an independent reproduction: Krawczynski et al. 2022 (Science, Tables S1-S2) report the answer, so their published numbers are the external BENCHMARK to compare against, NEVER a target to match. Report the real measured PD, PA, errors, MDP99, significance and fit parameters exactly as produced, and flag any quantity discrepant with the paper at >3 sigma as an honest, valuable result — do NOT tune arguments, hand-roll Stokes arithmetic, or back-fill any value to make this work agree with the paper. Only standard-measurement polarization products are in scope; the paper's corona-geometry/ray-tracing modelling is not.
- An honest failed/needs_human outcome (e.g. a required data file missing, or a capability that cannot be made available) is an acceptable, correct result — never fabricate a polarization measurement or an agreement with the paper. If the environment is broken, abort and report.
""",
)

repro_run = await run_research_nohup(repro_request, chatter=7, human_channel="telegram")

REPRO_LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/P12-cygx1-repro/step-7")
REPRO_LOG_DIR.mkdir(parents=True, exist_ok=True)
repro_md = REPRO_LOG_DIR / f"p12_cygx1_repro_krawczynski_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.md"
repro_run.monitor.write_markdown(repro_md)
repro_md, repro_run.verdict


[21:03:40] #0001 run L1 run.start: Research run started | run_id=bf1e39e079c7
[21:03:40] #0002 librarian L3 librarian.literature.start: Literature node started
[21:03:51] #0003 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_call librarian_io: librarian_io calling search_literature | tool=search_literature status=ok
[21:04:04] #0004 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_result librarian_io: librarian_io tool search_literature returned | tool=search_literature status=ok chars=860 figures=0
[21:04:04] #0005 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_call librarian_io: librarian_io calling search_literature | tool=search_literature status=ok
[21:04:07] #0006 librarian(claude-sonnet-4-6) L7 librarian_io.gather_literature.tool_result librarian_io: librarian_io tool search_literature returned | tool=search_literature status=ok chars=1388 figures=0
[21:04:07] #0007 librarian(claude-sonnet-4-6) L7 librarian_io.gather_li

(PosixPath('/data/fzhou/auto_research/logs/dev/P12-cygx1-repro/step-7/p12_cygx1_repro_krawczynski_20260710T093708_monitor.md'),
 ReviewVerdict(accepted=True, rationale='The analyst report is a genuine, well-executed independent reproduction and its results agree with the literature within stated uncertainties. Verification performed: (1) I inspected both analyst workers\' run memories — worker_1\'s header dump confirms real Level-2 event files for ObsID 01002901 (RA_OBJ=299.59032, DEC_OBJ=35.20161, TSTART=169402263/TSTOP=169926652, MJDREFI=57754, MJDREFF=0.00080074074074, DU1/DU2/DU3), and worker_0\'s directory listing shows the full chain of real intermediate products (per-DU src/bkg xpselect event files, PCUBE FITS, PHA1/PHA1Q/PHA1U Stokes spectra with the CALDB ixpe_d1_obssim20220702_alpha075_v013.rmf response written into headers, plus 11-segment energy-resolved src/bkg PCUBEs matching the prescribed 11-bin grid). This confirms the numbers came from correctly parameterised ixpeobss